<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "What Predicts Health?" (Random Forest feature importance, ML Appendix)

The claim: Average Position (43%) and Impressions (32%) are the top predictors of Health Score.

Methodology question: Health Score is defined in the paper's own methodology section as a composite built directly from Position (30 pts) and Impressions (30 pts), among other components. So two of the "top predictors" are inputs the label is partially built from. My question, respectfully: if we removed the label's own components (Position, Impressions) from the feature set, what would the remaining features (CTR, Scroll Depth, etc.) show us predicts health independently? The paper does flag this risk in its own chart-read note ("read this as model behavior, not... standalone optimization order"), which is good practice — a stronger version might report the model twice: once with label components included, once without.

Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The claim: a Logistic Regression model reaches 71% holdout accuracy separating growing from declining pages.

Methodology question: no base rate for growing vs. declining pages is reported next to this number. 71% only tells us something once we know what a naive "always predict the majority class" guess would score. If growing pages are, say, 62% of the sample, a model with zero real skill could already be close to that. My question: what is the class split in this holdout set, and how much of the 71% is actual separation versus just the base rate?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "pandas", "scikit-learn"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Same query as Week 5 — frozen, unchanged
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        LN(SUM(gsc_impressions) + 1) AS log_impressions,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(ga4_total_engagement_sec) AS total_engagement_sec,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
    ORDER BY content_hash_id
""").df()

df["ctr"] = df["total_clicks"] / df["total_impressions"]
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 50, 1000], labels=["1-3","4-10","11-20","21-50","50+"])
median_ctr_by_bucket = df.groupby("position_bucket", observed=True)["ctr"].transform("median")
df["is_underperforming"] = (df["ctr"] <= median_ctr_by_bucket).astype(int)

honest_features = ["avg_position", "log_impressions", "engaged_sessions", "total_engagement_sec", "scroll_events"]

print("Rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print("Base rate (is_underperforming=1):", df["is_underperforming"].mean().round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Unique clients: 47
Base rate (is_underperforming=1): 0.6027


In [2]:
# --- "BEFORE": naive random split (no grouping by client) ---
X = df[honest_features].fillna(0)
y = df["is_underperforming"]

X_train_rand, X_test_rand, y_train_rand, y_test_rand, train_idx_rand, test_idx_rand = train_test_split(
    X, y, df.index, test_size=0.3, random_state=42
)

model_rand = LogisticRegression(max_iter=1000, random_state=42)
model_rand.fit(X_train_rand, y_train_rand)
probs_rand = model_rand.predict_proba(X_test_rand)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p50_rand = precision_at_k(probs_rand, y_test_rand.values, 50)
base_rate_rand = y_test_rand.mean()

# Check: how many test rows share a client with a train row? (should be > 0 — this is the point)
train_clients_rand = df.loc[train_idx_rand, "client_hash_id"]
test_clients_rand = df.loc[test_idx_rand, "client_hash_id"]
overlap_rand = set(train_clients_rand) & set(test_clients_rand)

print("Naive random split — Precision@50:", round(p50_rand, 4), "| base rate:", round(base_rate_rand, 4))
print("Client overlap between train/test:", len(overlap_rand), "out of", df['client_hash_id'].nunique(), "total clients")

Naive random split — Precision@50: 1.0 | base rate: 0.6015
Client overlap between train/test: 45 out of 47 total clients


In [3]:
# --- "AFTER": honest split, grouped by client (same as Week 5) ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx_grp, test_idx_grp = next(splitter.split(df, groups=df["client_hash_id"]))

train_df_grp = df.iloc[train_idx_grp].reset_index(drop=True)
test_df_grp = df.iloc[test_idx_grp].reset_index(drop=True)

X_train_grp = train_df_grp[honest_features].fillna(0)
y_train_grp = train_df_grp["is_underperforming"]
X_test_grp = test_df_grp[honest_features].fillna(0)
y_test_grp = test_df_grp["is_underperforming"]

model_grp = LogisticRegression(max_iter=1000, random_state=42)
model_grp.fit(X_train_grp, y_train_grp)
probs_grp = model_grp.predict_proba(X_test_grp)[:, 1]

p50_grp = precision_at_k(probs_grp, y_test_grp.values, 50)
base_rate_grp = y_test_grp.mean()
overlap_grp = set(train_df_grp["client_hash_id"]) & set(test_df_grp["client_hash_id"])

comparison = pd.DataFrame({
    "split": ["Naive random (before)", "Grouped by client (after)"],
    "precision@50": [p50_rand, p50_grp],
    "base_rate": [base_rate_rand, base_rate_grp],
    "client_overlap": [len(overlap_rand), len(overlap_grp)]
})

print(comparison)

                       split  precision@50  base_rate  client_overlap
0      Naive random (before)          1.00   0.601543              45
1  Grouped by client (after)          0.98   0.556508               0


Before/after: the split changes what the score means, not just its value
The naive random split let 45 of 47 clients appear in both train and test, and Precision@50 came out at a perfect 1.00. The grouped split, which guarantees zero client overlap, gives Precision@50 of 0.98 — nearly identical, but honestly earned. The gap here is small (0.02), which is itself informative: it suggests this particular model isn't leaning heavily on client-specific memorization, since removing the overlap barely moved the score. That's a reassuring result, not a neutral one — a bigger drop would have meant the original grouped-split number from Week 5 was still hiding something. The grouped result remains the one that reflects performance on a client the model has never seen, which is the real deployment scenario, and it's the number that should be reported.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# Sanity check: does adding total_clicks (the label-source column) as a
# feature spike the score toward 1.0, proving our test harness can catch leakage?
leaky_features = honest_features + ["total_clicks"]

X_train_leak = train_df_grp[leaky_features].fillna(0)
X_test_leak = test_df_grp[leaky_features].fillna(0)

model_leak = LogisticRegression(max_iter=1000, random_state=42)
model_leak.fit(X_train_leak, y_train_grp)
probs_leak = model_leak.predict_proba(X_test_leak)[:, 1]

p50_leak = precision_at_k(probs_leak, y_test_grp.values, 50)
print("With total_clicks added (deliberately leaky):", round(p50_leak, 4))
print("Without it (honest, from Step 4):", round(p50_grp, 4))

With total_clicks added (deliberately leaky): 1.0
Without it (honest, from Step 4): 0.98


In [5]:
# --- Threshold test: does computing the median on train-only change the labels/score? ---

# Rebuild position_bucket + median CTR using ONLY the grouped-split train set
train_df_grp["position_bucket"] = pd.cut(train_df_grp["avg_position"], bins=[0, 3, 10, 20, 50, 1000], labels=["1-3","4-10","11-20","21-50","50+"])
train_only_median = train_df_grp.groupby("position_bucket", observed=True)["ctr"].median()

# Apply the TRAIN-ONLY threshold to the test set's own buckets
test_df_grp["position_bucket"] = pd.cut(test_df_grp["avg_position"], bins=[0, 3, 10, 20, 50, 1000], labels=["1-3","4-10","11-20","21-50","50+"])
test_df_grp["threshold_train_only"] = test_df_grp["position_bucket"].map(train_only_median)
test_df_grp["is_underperforming_v2"] = (test_df_grp["ctr"] <= test_df_grp["threshold_train_only"]).astype(int)

# Compare: how many test labels flipped between the original (all-data threshold)
# and this new train-only-threshold version?
flipped = (test_df_grp["is_underperforming_v2"] != y_test_grp).sum()
print("Test labels that changed:", flipped, "out of", len(test_df_grp))

# Re-score Precision@50 using the train-only-threshold labels
p50_train_only = precision_at_k(probs_grp, test_df_grp["is_underperforming_v2"].values, 50)
print("Precision@50 with train-only threshold:", round(p50_train_only, 4))
print("Precision@50 with original all-data threshold:", round(p50_grp, 4))

# Compare the actual median values, bucket by bucket
all_data_median = df.groupby("position_bucket", observed=True)["ctr"].median()
print("\nMedian CTR per bucket — train-only vs all-data:")
print(pd.DataFrame({"train_only": train_only_median, "all_data": all_data_median}))

Test labels that changed: 0 out of 45834
Precision@50 with train-only threshold: 0.98
Precision@50 with original all-data threshold: 0.98

Median CTR per bucket — train-only vs all-data:
                 train_only  all_data
position_bucket                      
1-3                     0.0       0.0
4-10                    0.0       0.0
11-20                   0.0       0.0
21-50                   0.0       0.0
50+                     0.0       0.0


Feature leakage audit: I checked the five final features individually. None is directly derived from is_underperforming. None uses the target itself. The features are aggregated from the available metrics, so I found no direct label-derived feature leakage. I also tested the known label-source column total_clicks separately; adding it increased Precision@50 from 0.98 to 1.00, confirming that the leakage check can detect an obviously unsafe feature.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest claim (from Week 5):

"engaged_sessions and log_impressions are by far the strongest signals, both pushing toward 'not underperforming' as engagement and traffic increase — a sensible relationship, since genuinely engaged visitors are more likely to have also clicked through."

Rewritten in safe language:

In this model, engaged_sessions and log_impressions had the largest coefficient magnitudes, both associated with a lower predicted probability of underperformance. This is an observed statistical association within the honest-split test set, not a causal claim — the model measures that engagement and underperformance move together in this data, not that engagement directly prevents underperformance. The direction is at least directionally consistent with a plausible mechanism (engaged visitors likely also click), but that mechanism isn't tested here.

Second claim (from Week 5):

"The baseline slightly outperforming the model makes sense here: my rule's underlying signal is essentially the same information used to build the label itself, so a simple, transparent rule already captures nearly all of it."

Rewritten in safe language:

The baseline measured a slightly higher Precision@50 (1.00) than the model (0.98) on the same honest-split test set. One plausible explanation is that the baseline's CTR-vs-bucket-median signal overlaps closely with how the label itself was constructed — but this is a hypothesis about why, not a measured finding on its own.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.